In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from patsy import dmatrix
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [20]:
don = pd.read_csv(r"C:\Users\cepe-s3-08\Desktop\fbach\SAh.csv", header=0, sep=",")
don.columns = [*don.columns[:-1], 'Y']
don.head(3)

,sbp,tobacco,ldl,adiposity,famhist,typea,obesity,alcohol,age,Y
0,160,12.00,5.73,23.11,Present,49,25.30,97.20,52,1
1,144,0.01,4.41,28.61,Absent,55,28.87,2.06,63,1
2,118,0.08,3.48,32.28,Present,52,29.14,3.81,46,0


In [21]:
#Préparation de la VC manuelle pour comparer les algos

In [32]:
Y = don["Y"].to_numpy()
nb=10
tmp = np.arange(don.shape[0])%nb
rng = np.random.default_rng(seed=1234)
bloc = rng.choice(tmp,size=don.shape[0],replace=False)
PROB = pd.DataFrame({"bloc":bloc,"Y":don["Y"],"log":0.0,
                    "ridge":0.0,"lasso":0.0,"elast":0.0,"bagging":0.0,"bagging_tree":0.0})

In [33]:
nomsvar = list(don.columns.difference(["Y"]))
#design matrix
formule = "~" + "+".join(nomsvar)
#formule = "~ " + " + ".join([f'Q("{v}")' for v in nomsvar])
dsX = dmatrix(formule,don)
X = np.asarray(dsX)[:,1:]

Il faut définir les grilles pour la régularisation

In [34]:
##################################################################
# LES GRILLES
def grille(X, y, type = "lasso", ng=400):
    scalerX = StandardScaler().fit(X)
    Xcr= scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y-y.mean()))).max()/X.shape[0]
    llc = np.linspace(0,-4,ng)
    ll = l0*10**llc
    if type=="lasso":
        Cs = 1/ 0.9/ X.shape[0] / (l0*10**(llc))
    elif type=="ridge":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 100)
    elif type=="enet":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 2)
    return Cs
#######################################################################

In [35]:
cr = StandardScaler()
for i in np.arange(nb):
    print(i)
    Xapp = X[bloc!=i,:]
    Xtest = X[bloc==i,:]
    Yapp = don[bloc!=i]["Y"]
    Ytest = don[bloc==i]["Y"]
    #### logistique
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[PROB.bloc==i,"log"] = log.predict_proba(Xtest)[:,1]
    ###################################################################
    ###ICI pb pour le choix de la grille de lambda
    ###################################################################
    ## lasso
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,Cs=Cs_lasso, solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[PROB.bloc==i,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ## ridge
    Cs_ridge = grille(Xapp,Yapp, "ridge")
    ridgecv =  LogisticRegressionCV(cv=10, penalty="l2", n_jobs=10,Cs=Cs_ridge,  solver="saga", max_iter=2000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[PROB.bloc==i,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]
    ## elas
    Cs_enet = grille(Xapp,Yapp, "enet")
    enetcv =  enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[PROB.bloc==i,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1]
    
    ## bagging
    # 1. On définit le modèle de base (ex: Régression Logistique ou Arbre)
    # Si vous voulez faire du bagging sur une logistique :
    base_model = LogisticRegression(solver="saga", max_iter=2000)

    # 2. On définit le BaggingClassifier
    # n_estimators=100 signifie qu'on va entraîner 100 modèles sur 100 échantillons bootstrap
    bagging_model = BaggingClassifier(estimator=base_model, 
                                  n_estimators=100, 
                                  n_jobs=-1, 
                                  random_state=42)

    # 3. Intégration dans le Pipeline (avec votre objet 'cr' pour le scaling)
    pipe_bagging = Pipeline(steps=[("cr", cr), ("bagging", bagging_model)])

    # 4. Entraînement et Prédiction
    pipe_bagging.fit(Xapp, Yapp)
    PROB.loc[PROB.bloc == i, "bagging"] = pipe_bagging.predict_proba(Xtest)[:, 1]
    

    # 1. Définition de l'arbre de base
    # On le laisse souvent "profond" (sans max_depth) pour que le Bagging fasse son travail
    base_tree = DecisionTreeClassifier()

    # 2. Configuration du Bagging
    # n_estimators=100 : on crée 100 arbres différents par bootstrap
    bagging_tree = BaggingClassifier(
        estimator=base_tree, 
        n_estimators=100, 
        max_samples=0.8,  # Chaque arbre voit 80% des données (augmente la diversité)
        n_jobs=-1, 
        random_state=42
    )

    # 3. Pipeline et Fit
    pipe_bagging_tree = Pipeline(steps=[("cr", cr), ("bagging_tree", bagging_tree)])
    pipe_bagging_tree.fit(Xapp, Yapp)

    # 4. Prédiction des probabilités
    PROB.loc[PROB.bloc == i, "bagging_tree"] = pipe_bagging_tree.predict_proba(Xtest)[:, 1]
    

0
1
2
3
4
5
6
7
8
9


In [36]:
PROB.head(3)

,bloc,Y,log,ridge,lasso,elast,bagging,bagging_tree
0,5,1,0.629664,0.583829,0.595181,0.590338,0.624340,0.78
1,5,1,0.355473,0.361561,0.417798,0.396662,0.374400,0.30
2,8,0,0.256772,0.288785,0.292447,0.291543,0.248193,0.11


Je veux calculer le taux d'erreur

In [37]:
import sklearn.metrics as sklm
mc = pd.Series(0.0, index=PROB.columns[1:])
s = 0.5
for i in range(mc.shape[0]):
    mc.iloc[i] = sklm.zero_one_loss(PROB.Y, PROB.iloc[:,i+1]>s)
round(mc,3)

Y               0.000
log             0.281
ridge           0.273
lasso           0.273
elast           0.271
bagging         0.279
bagging_tree    0.316
dtype: float64

Mais je peux aussi calculer d'autres critères

In [38]:
noms = PROB.columns[1:]
matsB = pd.DataFrame({"seuil": pd.Series(0.0, index=noms)})
s = .5
for nom in noms:
    matsB.loc[nom,"seuil"] = s
    confmat = sklm.confusion_matrix(PROB.Y, PROB.loc[:,nom]>=s)
    print(confmat)
    matsB.loc[nom, "tn"] = confmat[0,0]
    matsB.loc[nom, "tp"] = confmat[1,1]
    matsB.loc[nom, "fn"] = confmat[1,0]
    matsB.loc[nom, "fp"] = confmat[0,1]
    matsB.loc[nom,"sensitivity"] = confmat[1,1]/(confmat[1,1]+confmat[1,0])
    matsB.loc[nom,"specificity"] = confmat[0,0]/(confmat[0,0]+confmat[0,1])
    matsB.loc[nom,"accuracy"] = sklm.accuracy_score(PROB.Y, PROB.loc[:,nom]>=s)
print(matsB.round(3))

[[302   0]
 [  0 160]]
[[251  51]
 [ 79  81]]
[[263  39]
 [ 87  73]]
[[263  39]
 [ 87  73]]
[[267  35]
 [ 90  70]]
[[251  51]
 [ 78  82]]
[[240  62]
 [ 90  70]]
              seuil     tn     tp    fn    fp  sensitivity  specificity  \
Y               0.5  302.0  160.0   0.0   0.0        1.000        1.000   
log             0.5  251.0   81.0  79.0  51.0        0.506        0.831   
ridge           0.5  263.0   73.0  87.0  39.0        0.456        0.871   
lasso           0.5  263.0   73.0  87.0  39.0        0.456        0.871   
elast           0.5  267.0   70.0  90.0  35.0        0.438        0.884   
bagging         0.5  251.0   82.0  78.0  51.0        0.512        0.831   
bagging_tree    0.5  240.0   70.0  90.0  62.0        0.438        0.795   

              accuracy  
Y                1.000  
log              0.719  
ridge            0.727  
lasso            0.727  
elast            0.729  
bagging          0.721  
bagging_tree     0.671  


Je peux aussi calculer le seuil qui respecte les proportions

In [39]:
matsN = pd.DataFrame({"seuil": pd.Series(0.0, index=noms)})
nbr0 = don.Y.value_counts()[0]
for nom in noms:
    tmp = PROB.loc[:,nom].sort_values(ascending=True)
    s = (tmp.iloc[nbr0-1]+tmp.iloc[nbr0])/2
    confmat = sklm.confusion_matrix(PROB.Y, PROB.loc[:,nom]>=s)
    matsN.loc[nom,"seuil"] = s
    matsN.loc[nom, "tn"] = confmat[0,0]
    matsN.loc[nom, "tp"] = confmat[1,1]
    matsN.loc[nom, "fn"] = confmat[1,0]
    matsN.loc[nom, "fp"] = confmat[0,1]
    matsN.loc[nom,"sensitivity"] = confmat[1,1]/(confmat[1,1]+confmat[1,0])
    matsN.loc[nom,"specificity"] = confmat[0,0]/(confmat[0,0]+confmat[0,1])
    matsN.loc[nom,"accuracy"] = sklm.accuracy_score(PROB.Y, PROB.loc[:,nom]>=s)
print(matsN.round(3))

              seuil     tn     tp    fn    fp  sensitivity  specificity  \
Y             0.500  302.0  160.0   0.0   0.0        1.000        1.000   
log           0.435  236.0   94.0  66.0  66.0        0.588        0.781   
ridge         0.420  237.0   95.0  65.0  65.0        0.594        0.785   
lasso         0.430  236.0   94.0  66.0  66.0        0.588        0.781   
elast         0.424  236.0   94.0  66.0  66.0        0.588        0.781   
bagging       0.438  236.0   94.0  66.0  66.0        0.588        0.781   
bagging_tree  0.445  226.0   84.0  76.0  76.0        0.525        0.748   

              accuracy  
Y                1.000  
log              0.714  
ridge            0.719  
lasso            0.714  
elast            0.714  
bagging          0.714  
bagging_tree     0.671  
